In [171]:
import tskit
import pandas as pd
import numpy as np
import os
import cyvcf2

In [172]:
singer_mut = pd.read_table("/home/jobste/Chr14_NonHybrid_3_4_muts_99.txt", header=None, sep=" ", names = ["Pos", "Col2", "Col3", "Anc"])


In [173]:
singer_mut.Anc.value_counts()

Anc
1    52462
0     7130
Name: count, dtype: int64

In [174]:
index = pd.read_table("/home/jobste/Chr14_NonHybrid_diploid_polymorphic.index", header=None)

In [175]:
index

,0,1
0,0,5412
1,1000000,30533837
2,2000000,94746171
3,3000000,153126916
4,4000000,214647047
5,5000000,278397703
6,6000000,323255948
7,7000000,371593304
8,8000000,429345434
9,9000000,480736237


In [176]:
singer_mut.loc[:, "POS"] = singer_mut.Pos + index.iloc[3][0]

In [177]:
vcf = "/home/jobste/Data/Chr14_imputed_AllInfo.vcf.gz"

In [178]:
command = "bcftools view -r 14:" + str(index.iloc[3][0]) + "-" + str(index.iloc[4][0]) + " " + vcf  + " > " + vcf.strip(".vcf.gz") + "_Chunk3.vcf"

In [179]:
command

'bcftools view -r 14:3000000-4000000 /home/jobste/Data/Chr14_imputed_AllInfo.vcf.gz > /home/jobste/Data/Chr14_imputed_AllInfo_Chunk3.vcf'

In [180]:
os.system(command)

0

In [181]:
vcfChunk = cyvcf2.VCF(vcf.strip(".vcf.gz") + "_Chunk3.vcf")

In [182]:
ancestralList = []
ancestralTypeList = []
pos = []
for variant in vcfChunk:
    pos.append(variant.POS)
    if variant.INFO["Prob"] >= 0.9:
        ancestral = variant.INFO["Major"]
        ancestralType = "Major"
    elif (variant.INFO["Prob"] < 0.1) & (variant.INFO["Prob"] != -99):
        al = [variant.REF, variant.ALT[0]]
        ancestral = al[al != variant.INFO["Major"]]
        ancestralType = "Minor"
    else:
        if variant.INFO["MajorOutgroup"] != '0':
            ancestral = variant.INFO["MajorOutgroup"]
            if ancestral == variant.INFO["Major"]:
                ancestralType = "MajorOutgroup_Major"
            else:
                ancestralType = "MajorOutgroup_Minor"
        else:
            ancestral = 0
            ancestralType = "0"

    ancestralList.append(ancestral)
    ancestralTypeList.append(ancestralType)

In [183]:
len(ancestralTypeList)

59015

In [184]:
ancestralDF = pd.DataFrame({"POS": pos, "AncAl": ancestralList, "AncType": ancestralTypeList})

In [185]:
ancestralDF.shape

(59015, 3)

In [186]:
sum(ancestralDF.POS.duplicated())

0

In [187]:
singer_mut = singer_mut[~singer_mut.POS.isin(singer_mut[singer_mut.POS.duplicated()].POS)]

In [193]:
singer_mut.Anc.value_counts()

Anc
1    43926
Name: count, dtype: int64

In [188]:
singer_mut_anc = ancestralDF.merge(singer_mut, how = "left", on = "POS")

In [191]:
singer_mut_anc.Anc.value_counts()

Anc
1.0    43926
Name: count, dtype: int64

In [99]:
from collections import Counter
Counter(singer_mut[singer_mut.Anc == 0].AncType)

Counter({'MajorOutgroup_Minor': 2885,
         'Major': 2609,
         '0': 1280,
         'Minor': 309,
         'MajorOutgroup_Major': 47})

In [102]:
num = 0
acs = []
for variant in vcfChunk:
    if variant.INFO["Major"] == ancestralList[num]:
        acs.append(variant.INFO["AC"])
    else:
        if ancestralList[num] == 0:
            acs.append("Skip")
        else:
            acs.append(variant.INFO["AN"] - variant.INFO["AC"])
    num += 1


,Pos,Col2,Col3,Anc,POS,AncAl,AncType
0,13,2849,119377,1,3000013,0,0
1,44,100653,124941,1,3000044,C,Major
2,170,976,96147,1,3000170,C,Major
3,203,98405,115882,1,3000203,C,Major
4,210,91621,117733,1,3000210,C,Major
